# MGCI Data Export Pipeline

**SDG Indicator 15.4.2: Mountain Green Cover Index**

This notebook exports Sentinel-2 imagery and terrain data from Google Earth Engine for the MGCI deep learning pipeline.

### Outputs
- Sentinel-2 median composite (B2, B3, B4, B8)
- DEM elevation and slope data
- NDVI-derived vegetation labels

**Study Area:** Jazan Province, Saudi Arabia  
**Resolution:** 10m  
**Temporal Range:** 2024

## 1. Setup and Configuration

In [ ]:
!pip install -q earthengine-api geemap folium
print("Dependencies installed")

In [ ]:
import ee
import geemap
import folium
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

print(f"Libraries imported - {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Authenticate and initialize Earth Engine
ee.Authenticate()

PROJECT_ID = 'your-project-id'  # <-- Update with your GEE project ID
ee.Initialize(project=PROJECT_ID)

print(f"Earth Engine initialized (Project: {PROJECT_ID})")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
class Config:
    """
    Configuration parameters for MGCI data export.
    Update these settings for your study region.
    """
    
    # Region settings (FAO GAUL administrative boundaries)
    GAUL_LEVEL = 1                    # 0=Country, 1=Province, 2=District
    GAUL_FIELD = 'ADM1_NAME'
    REGION_NAME = 'Jizan'
    COUNTRY = 'Saudi Arabia'

    # Time period
    YEAR = 2024
    START_DATE = '2024-01-01'
    END_DATE = '2024-12-31'

    # Vegetation threshold (climate-dependent)
    # 0.20: Very humid tropical | 0.25: Humid subtropical
    # 0.30: Semi-arid          | 0.35: Arid
    NDVI_THRESHOLD = 0.30
    CLIMATE_ZONE = 'Semi-arid'

    # Patch settings
    PATCH_SIZE = 256                  # pixels (256x256)
    RESOLUTION = 10                   # meters

    # Filtering criteria
    MAX_CLOUD_COVER = 20              # maximum cloud cover %
    MOUNTAIN_THRESHOLD = 300          # minimum elevation (m) per Kapos Class 6
    MIN_MOUNTAIN_PCT = 0.01           # minimum 1% mountain coverage per patch

    # Export settings
    EXPORT_FOLDER = 'MGCI_Jizan_2024'
    CRS = 'EPSG:32637'                # UTM Zone 37N

    # Testing mode (set True for quick test with limited patches)
    TEST_MODE = False
    TEST_PATCHES = 10


# Display configuration
print(f"Region: {Config.REGION_NAME}, {Config.COUNTRY}")
print(f"Period: {Config.START_DATE} to {Config.END_DATE}")
print(f"Climate Zone: {Config.CLIMATE_ZONE}")
print(f"NDVI Threshold: {Config.NDVI_THRESHOLD}")
print(f"Patch Size: {Config.PATCH_SIZE}x{Config.PATCH_SIZE} pixels")
print(f"Resolution: {Config.RESOLUTION}m")
print(f"Mountain Threshold: {Config.MOUNTAIN_THRESHOLD}m")
print(f"Export Folder: {Config.EXPORT_FOLDER}")

## 2. Load Study Region

In [ ]:
print(f"Loading region: {Config.REGION_NAME}...")

# Load FAO GAUL administrative boundaries
gaul = ee.FeatureCollection(f'FAO/GAUL/2015/level{Config.GAUL_LEVEL}')
region = gaul.filter(ee.Filter.eq(Config.GAUL_FIELD, Config.REGION_NAME))
geometry = region.geometry()

# Get region statistics
area_km2 = geometry.area(100).divide(1e6).getInfo()
bounds = geometry.bounds().getInfo()['coordinates'][0]
centroid = geometry.centroid().getInfo()['coordinates']

print(f"Region loaded successfully")
print(f"  Area: {area_km2:,.2f} km²")
print(f"  Centroid: {centroid[1]:.4f}°N, {centroid[0]:.4f}°E")

In [ ]:
print("Loading DEM and calculating terrain...")

# Buffer geometry to avoid edge effects
geometry_buffered = geometry.buffer(10000)

# Load Copernicus DEM
dem_raw = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
    .filterBounds(geometry_buffered) \
    .select('DEM') \
    .mosaic()

# Reproject to UTM for accurate slope calculation (ensures X, Y, Z units match)
dem_projected = dem_raw.reproject(crs=Config.CRS, scale=30)

# Calculate slope on projected DEM
slope_projected = ee.Terrain.slope(dem_projected)

# Clip to region
dem = dem_projected.clip(geometry).unmask(0).rename('elevation')
slope = slope_projected.clip(geometry).unmask(0).rename('slope')

# Create mountain mask (elevation >= threshold)
mountain_mask = dem.gte(Config.MOUNTAIN_THRESHOLD)

print(f"DEM and slope calculated using CRS: {Config.CRS}")

## 3. Load and Process Satellite Data

In [ ]:
print("Verifying slope data...")

# Get elevation statistics
elev_stats = dem.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
    geometry=geometry, scale=100, maxPixels=1e9
).getInfo()

# Get slope statistics
slope_stats = slope.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True).combine(ee.Reducer.stdDev(), '', True),
    geometry=geometry, scale=100, maxPixels=1e9
).getInfo()

elev_min = elev_stats.get('elevation_min', 0) or 0
elev_max = elev_stats.get('elevation_max', 0) or 0
elev_mean = elev_stats.get('elevation_mean', 0) or 0

slope_min = slope_stats.get('slope_min', 0) or 0
slope_max = slope_stats.get('slope_max', 0) or 0
slope_mean = slope_stats.get('slope_mean', 0) or 0
slope_std = slope_stats.get('slope_stdDev', 0) or 0

print(f"Elevation: {elev_min:.0f}m - {elev_max:.0f}m (mean: {elev_mean:.0f}m)")
print(f"Slope: {slope_min:.2f}° - {slope_max:.2f}° (mean: {slope_mean:.2f}°, std: {slope_std:.2f}°)")

# Validation
if slope_max < 1:
    print("ERROR: Slope data is missing - check DEM loading")
    SLOPE_VALID = False
elif slope_max < 15:
    print(f"WARNING: Slope values seem low ({slope_max:.1f}° max)")
    SLOPE_VALID = True
else:
    print(f"Slope data verified (max {slope_max:.1f}° is reasonable for mountains)")
    SLOPE_VALID = True

In [ ]:
print("Loading Sentinel-2 imagery...")

def mask_clouds(image):
    """Mask clouds using QA60 and SCL bands."""
    qa60 = image.select('QA60')
    cloud_mask = qa60.bitwiseAnd(1 << 10).eq(0).And(qa60.bitwiseAnd(1 << 11).eq(0))
    scl = image.select('SCL')
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(cloud_mask.And(scl_mask))

# Load Sentinel-2 Surface Reflectance
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(geometry) \
    .filterDate(Config.START_DATE, Config.END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', Config.MAX_CLOUD_COVER)) \
    .map(mask_clouds)

image_count = s2.size().getInfo()

# Create median composite (more realistic than max-NDVI composite)
composite = s2.median().clip(geometry).unmask(0)

print(f"Sentinel-2 median composite created")
print(f"  Images used: {image_count}")
print(f"  Date range: {Config.START_DATE} to {Config.END_DATE}")
print(f"  Cloud filter: < {Config.MAX_CLOUD_COVER}%")

In [ ]:
print("Calculating vegetation indices...")

# Calculate NDVI
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('ndvi')

# Binary vegetation label
veg_label = ndvi.gt(Config.NDVI_THRESHOLD).unmask(0).rename('vegetation')

# Get statistics
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
    geometry=geometry, scale=100, maxPixels=1e9
).getInfo()

veg_pct = veg_label.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=geometry, scale=100, maxPixels=1e9
).get('vegetation').getInfo() * 100

print(f"NDVI range: {ndvi_stats.get('ndvi_min', 0):.3f} to {ndvi_stats.get('ndvi_max', 0):.3f}")
print(f"NDVI mean: {ndvi_stats.get('ndvi_mean', 0):.3f}")
print(f"Vegetation coverage (NDVI > {Config.NDVI_THRESHOLD}): {veg_pct:.1f}%")

In [ ]:
print("Creating export image stack...")

# Prepare bands with explicit conversion
blue = composite.select('B2').toFloat().unmask(0).rename('Blue')
green = composite.select('B3').toFloat().unmask(0).rename('Green')
red = composite.select('B4').toFloat().unmask(0).rename('Red')
nir = composite.select('B8').toFloat().unmask(0).rename('NIR')
elev_band = dem.toFloat().unmask(0).rename('Elevation')
slope_band = slope.toFloat().unmask(0).rename('Slope')
veg_band = veg_label.toFloat().unmask(0).rename('VegLabel')

# Stack all bands
export_image = blue.addBands(green).addBands(red).addBands(nir) \
    .addBands(elev_band).addBands(slope_band).addBands(veg_band)

print("Export image ready")
print("Band order:")
print("  [0] Blue       - Sentinel-2 B2")
print("  [1] Green      - Sentinel-2 B3")
print("  [2] Red        - Sentinel-2 B4")
print("  [3] NIR        - Sentinel-2 B8")
print("  [4] Elevation  - DEM (meters)")
print("  [5] Slope      - Degrees (0-90)")
print("  [6] VegLabel   - Binary (0/1)")

In [ ]:
print("Verifying export image bands...")

# Sample at region centroid
sample_values = export_image.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=geometry.centroid().buffer(1000),
    scale=10, maxPixels=1e6
).getInfo()

print("Sample values at region center:")
for band in ['Blue', 'Green', 'Red', 'NIR', 'Elevation', 'Slope', 'VegLabel']:
    value = sample_values.get(band, None)
    if value is not None:
        status = "OK" if (band != 'Slope' or value > 0) else "CHECK"
        print(f"  {band:12s}: {value:>10.2f} [{status}]")
    else:
        print(f"  {band:12s}: {'NULL':>10s} [ERROR]")

## 4. Interactive Map Visualization

In [ ]:
print("Creating interactive map...")

# Create map centered on region
Map = geemap.Map(center=[centroid[1], centroid[0]], zoom=9)

# Visualization parameters
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
vis_false = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 4000}
vis_elev = {'min': 0, 'max': 2500, 'palette': ['green', 'yellow', 'brown', 'white']}
vis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'yellow', 'orange', 'red', 'darkred']}
vis_ndvi = {'min': -0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'lightgreen', 'darkgreen']}
vis_veg = {'min': 0, 'max': 1, 'palette': ['white', 'green']}
vis_mtn = {'min': 0, 'max': 1, 'palette': ['white', 'saddlebrown']}

# Add layers
Map.addLayer(region.style(color='red', fillColor='00000000', width=2), {}, 'Region Boundary')
Map.addLayer(composite, vis_rgb, 'RGB Composite', True)
Map.addLayer(composite, vis_false, 'False Color (Vegetation=Red)', False)
Map.addLayer(dem, vis_elev, 'Elevation', False)
Map.addLayer(slope, vis_slope, 'Slope', False)
Map.addLayer(mountain_mask, vis_mtn, 'Mountain Mask', False)
Map.addLayer(ndvi, vis_ndvi, 'NDVI', False)
Map.addLayer(veg_label, vis_veg, 'Vegetation Label', False)

# Green mountain cover
green_mountain = veg_label.And(mountain_mask).selfMask()
Map.addLayer(green_mountain, {'palette': ['forestgreen']}, 'Green Mountain Cover', False)

Map.addLayerControl()

print("Map created - use layer control to toggle layers")
Map

## 5. Regional Statistics and MGCI Preview

In [ ]:
print("Calculating regional statistics...")

# Mountain area
mtn_area_km2 = mountain_mask.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry, scale=100, maxPixels=1e9
).get('elevation').getInfo() / 1e6

# Green mountain area
green_mtn = veg_label.And(mountain_mask)
green_mtn_km2 = green_mtn.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry, scale=100, maxPixels=1e9
).get('vegetation').getInfo() / 1e6

# Calculate preliminary MGCI
mgci_preliminary = (green_mtn_km2 / mtn_area_km2 * 100) if mtn_area_km2 > 0 else 0

print(f"\nRegional Statistics:")
print(f"  Total area: {area_km2:,.2f} km²")
print(f"  Mountain area (>={Config.MOUNTAIN_THRESHOLD}m): {mtn_area_km2:,.2f} km² ({mtn_area_km2/area_km2*100:.1f}%)")
print(f"  Green mountain area: {green_mtn_km2:,.2f} km²")
print(f"\n  Preliminary MGCI: {mgci_preliminary:.2f}%")
print(f"  (Note: Final MGCI will include slope correction)")

In [ ]:
# Visualize regional statistics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Area distribution
ax1 = axes[0, 0]
sizes = [green_mtn_km2, mtn_area_km2 - green_mtn_km2, area_km2 - mtn_area_km2]
labels = ['Green Mountain', 'Bare Mountain', 'Non-Mountain']
colors = ['#228B22', '#8B4513', '#D3D3D3']
ax1.pie(sizes, explode=(0.05, 0, 0), labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title(f'{Config.REGION_NAME} Area Distribution\n(Total: {area_km2:,.0f} km²)', fontweight='bold')

# MGCI gauge
ax2 = axes[0, 1]
ax2.barh(['MGCI'], [mgci_preliminary], color='forestgreen', height=0.5)
ax2.barh(['MGCI'], [100 - mgci_preliminary], left=[mgci_preliminary], color='lightgray', height=0.5)
ax2.set_xlim(0, 100)
ax2.set_xlabel('Percentage (%)')
ax2.set_title(f'Preliminary MGCI: {mgci_preliminary:.1f}%', fontweight='bold')
ax2.axvline(mgci_preliminary, color='red', linestyle='--', linewidth=2)

# Terrain statistics
ax3 = axes[0, 2]
ax3.text(0.5, 0.5, f"TERRAIN STATISTICS\n\n"
         f"Elevation Range:\n{elev_min:.0f}m - {elev_max:.0f}m\n\n"
         f"Slope Statistics:\n"
         f"Max: {slope_max:.1f}°\n"
         f"Mean: {slope_mean:.1f}°\n"
         f"Std: {slope_std:.1f}°",
         transform=ax3.transAxes, fontsize=12,
         verticalalignment='center', horizontalalignment='center',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax3.axis('off')
ax3.set_title('Terrain Statistics', fontweight='bold')

# Data quality
ax4 = axes[1, 0]
quality_items = [
    ('Sentinel-2 Images', f'{image_count}', 'OK'),
    ('Composite Type', 'MEDIAN', 'OK'),
    ('Slope Data', f'Max: {slope_max:.1f}°', 'OK' if SLOPE_VALID else 'ERROR'),
    ('NDVI Threshold', f'{Config.NDVI_THRESHOLD}', 'OK'),
    ('Resolution', f'{Config.RESOLUTION}m', 'OK'),
    ('Mountain Threshold', f'{Config.MOUNTAIN_THRESHOLD}m', 'OK')
]
table_data = [[item[0], item[1], item[2]] for item in quality_items]
table = ax4.table(cellText=table_data, colLabels=['Parameter', 'Value', 'Status'],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax4.axis('off')
ax4.set_title('Data Quality Check', fontweight='bold')

# Area breakdown
ax5 = axes[1, 1]
categories = ['Total', 'Mountain', 'Green Mtn']
values = [area_km2, mtn_area_km2, green_mtn_km2]
bar_colors = ['steelblue', 'saddlebrown', 'forestgreen']
bars = ax5.bar(categories, values, color=bar_colors, edgecolor='black')
ax5.set_ylabel('Area (km²)')
ax5.set_title('Area Breakdown', fontweight='bold')
for bar, val in zip(bars, values):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + area_km2*0.02,
             f'{val:,.0f}', ha='center', fontsize=10)

# Export summary
ax6 = axes[1, 2]
summary_text = f"""
EXPORT CONFIGURATION
─────────────────────────────────
Region:         {Config.REGION_NAME}
Year:           {Config.YEAR}
Resolution:     {Config.RESOLUTION}m
Patch Size:     {Config.PATCH_SIZE}x{Config.PATCH_SIZE}
Export Folder:  {Config.EXPORT_FOLDER}

BAND ORDER
─────────────────────────────────
[0] Blue      [4] Elevation
[1] Green     [5] Slope
[2] Red       [6] VegLabel
[3] NIR
"""
ax6.text(0.1, 0.5, summary_text, transform=ax6.transAxes, fontsize=11,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))
ax6.axis('off')
ax6.set_title('Export Configuration', fontweight='bold')

plt.suptitle(f'MGCI Data Export Summary - {Config.REGION_NAME}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/data_export_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("Summary saved to /content/data_export_summary.png")

## 6. Sample Patch Preview

In [ ]:
print("Generating sample patch preview...")

# Sample point in high mountain area
sample_point = ee.Geometry.Point([43.085, 17.265])

# Define patch bounds
patch_size_m = Config.PATCH_SIZE * Config.RESOLUTION
sample_patch = sample_point.buffer(patch_size_m / 2).bounds()

try:
    # Reproject for visualization
    preview_image = export_image.reproject(crs=Config.CRS, scale=Config.RESOLUTION)
    sample_data = preview_image.sampleRectangle(region=sample_patch, defaultValue=0).getInfo()

    # Extract bands
    bands = {}
    for name in ['Blue', 'Green', 'Red', 'NIR', 'Elevation', 'Slope', 'VegLabel']:
        bands[name] = np.array(sample_data['properties'][name])

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    # RGB
    rgb = np.stack([bands['Red'], bands['Green'], bands['Blue']], axis=-1)
    rgb = np.clip(rgb / 2500, 0, 1)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('RGB Composite', fontweight='bold')
    axes[0, 0].axis('off')

    # False Color
    fc = np.stack([bands['NIR'], bands['Red'], bands['Green']], axis=-1)
    fc = np.clip(fc / 3000, 0, 1)
    axes[0, 1].imshow(fc)
    axes[0, 1].set_title('False Color (NIR-R-G)', fontweight='bold')
    axes[0, 1].axis('off')

    # Elevation
    im = axes[0, 2].imshow(bands['Elevation'], cmap='terrain')
    axes[0, 2].set_title(f"Elevation ({bands['Elevation'].min():.0f}-{bands['Elevation'].max():.0f}m)", fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046)

    # Slope
    im = axes[0, 3].imshow(bands['Slope'], cmap='YlOrRd', vmin=0, vmax=45)
    axes[0, 3].set_title(f"Slope (max={bands['Slope'].max():.1f}°)", fontweight='bold')
    axes[0, 3].axis('off')
    plt.colorbar(im, ax=axes[0, 3], fraction=0.046)

    # NDVI
    nir = bands['NIR'].astype(float)
    red = bands['Red'].astype(float)
    ndvi_arr = np.divide((nir - red), (nir + red), out=np.zeros_like(nir), where=(nir+red)!=0)
    im = axes[1, 0].imshow(ndvi_arr, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    axes[1, 0].set_title(f'NDVI (mean={ndvi_arr.mean():.3f})', fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)

    # Vegetation Label
    axes[1, 1].imshow(bands['VegLabel'], cmap='Greens', vmin=0, vmax=1)
    veg_pct_patch = bands['VegLabel'].mean() * 100
    axes[1, 1].set_title(f'Vegetation Label ({veg_pct_patch:.1f}%)', fontweight='bold')
    axes[1, 1].axis('off')

    # Mountain Mask
    mtn_mask = bands['Elevation'] >= Config.MOUNTAIN_THRESHOLD
    axes[1, 2].imshow(mtn_mask, cmap='copper', vmin=0, vmax=1)
    mtn_pct_patch = mtn_mask.mean() * 100
    axes[1, 2].set_title(f'Mountain Mask ({mtn_pct_patch:.1f}%)', fontweight='bold')
    axes[1, 2].axis('off')

    # MGCI Visualization
    green_mtn_patch = mtn_mask & (bands['VegLabel'] == 1)
    overlay = np.zeros((*bands['Elevation'].shape, 3))
    overlay[~mtn_mask] = [0.8, 0.8, 0.8]
    overlay[mtn_mask & ~green_mtn_patch] = [0.6, 0.4, 0.2]
    overlay[green_mtn_patch] = [0.1, 0.7, 0.1]

    axes[1, 3].imshow(overlay)
    patch_mgci = green_mtn_patch.sum() / max(mtn_mask.sum(), 1) * 100
    axes[1, 3].set_title(f'MGCI: {patch_mgci:.1f}%', fontweight='bold')
    axes[1, 3].axis('off')

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=[0.8, 0.8, 0.8], label='Non-Mountain'),
        Patch(facecolor=[0.6, 0.4, 0.2], label='Bare Mountain'),
        Patch(facecolor=[0.1, 0.7, 0.1], label='Green Mountain')
    ]
    axes[1, 3].legend(handles=legend_elements, loc='lower right', fontsize=8)

    plt.suptitle('Sample Patch Preview (High Elevation Area)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/sample_patch_preview.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Preview saved to /content/sample_patch_preview.png")
    print(f"\nPatch Statistics:")
    print(f"  Slope Max: {bands['Slope'].max():.1f}°")
    print(f"  Elevation Range: {bands['Elevation'].min():.0f} - {bands['Elevation'].max():.0f}m")
    print(f"  Vegetation: {veg_pct_patch:.1f}%")
    print(f"  Mountain: {mtn_pct_patch:.1f}%")
    print(f"  Patch MGCI: {patch_mgci:.1f}%")

except Exception as e:
    print(f"Could not generate preview: {e}")
    print("This may be a timeout issue. The actual export will likely still work.")

## 7. Generate Export Patches

In [ ]:
print("Generating patch grid...")

# Get bounds
bounds_info = geometry.bounds().getInfo()['coordinates'][0]
min_x = min(p[0] for p in bounds_info)
max_x = max(p[0] for p in bounds_info)
min_y = min(p[1] for p in bounds_info)
max_y = max(p[1] for p in bounds_info)

# Calculate patch size in degrees (approximate)
patch_size_deg = (Config.PATCH_SIZE * Config.RESOLUTION) / 111000

# Generate grid
patches = []
y = min_y
while y < max_y:
    x = min_x
    while x < max_x:
        patch_geom = ee.Geometry.Rectangle([x, y, x + patch_size_deg, y + patch_size_deg])
        patches.append({'geometry': patch_geom, 'x': x, 'y': y})
        x += patch_size_deg
    y += patch_size_deg

print(f"Generated {len(patches)} potential patches")

In [ ]:
print("Filtering patches for mountain coverage...")

def check_mountain_coverage(patch_info):
    """Check if patch has minimum mountain coverage."""
    try:
        mtn_frac = mountain_mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=patch_info['geometry'],
            scale=Config.RESOLUTION,
            maxPixels=1e6
        ).get('elevation').getInfo()
        return mtn_frac is not None and mtn_frac >= Config.MIN_MOUNTAIN_PCT
    except:
        return False

# Filter patches
export_patches = []
for i, patch in enumerate(patches):
    if check_mountain_coverage(patch):
        export_patches.append(patch)
    
    if (i + 1) % 100 == 0:
        print(f"  Checked {i+1}/{len(patches)} patches, found {len(export_patches)} valid...")

print(f"\nFound {len(export_patches)} patches with >= {Config.MIN_MOUNTAIN_PCT*100:.0f}% mountain coverage")

# Apply test mode limit if enabled
if Config.TEST_MODE:
    export_patches = export_patches[:Config.TEST_PATCHES]
    print(f"Test mode: limited to {len(export_patches)} patches")

## 8. Export to Google Drive

In [ ]:
print(f"Starting export of {len(export_patches)} patches...")
print(f"Destination: Google Drive/{Config.EXPORT_FOLDER}")

tasks = []
for i, patch in enumerate(export_patches):
    # Create filename
    filename = f"patch_{i:04d}"
    
    # Create export task
    task = ee.batch.Export.image.toDrive(
        image=export_image,
        description=filename,
        folder=Config.EXPORT_FOLDER,
        region=patch['geometry'],
        scale=Config.RESOLUTION,
        crs=Config.CRS,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    tasks.append(task)

    if (i + 1) % 50 == 0 or (i + 1) == len(export_patches):
        print(f"  Started {i+1}/{len(export_patches)} tasks...")

print(f"\nStarted {len(tasks)} export tasks")
print(f"Monitor progress at: https://code.earthengine.google.com/tasks")

In [ ]:
# Check export status (run multiple times to refresh)
print("Checking export status...")
print("(Run this cell again to refresh)\n")

completed = sum(1 for t in tasks if t.status()['state'] == 'COMPLETED')
running = sum(1 for t in tasks if t.status()['state'] in ['RUNNING', 'READY'])
failed = sum(1 for t in tasks if t.status()['state'] == 'FAILED')

pct = completed / len(tasks) * 100
bar = '█' * int(pct/2.5) + '░' * (40 - int(pct/2.5))

print(f"[{bar}] {pct:.1f}%")
print(f"\n  Completed: {completed}/{len(tasks)}")
print(f"  Running:   {running}/{len(tasks)}")
print(f"  Failed:    {failed}/{len(tasks)}")

if completed == len(tasks):
    print(f"\nAll exports complete!")
    print(f"\nNext step: Open Notebook 2 and set:")
    print(f"  DATA_DIR = '/content/drive/MyDrive/{Config.EXPORT_FOLDER}'")

## 9. Summary

This notebook exports the following data for MGCI computation:

| Band | Name | Description |
|------|------|-------------|
| 0 | Blue | Sentinel-2 B2 (490nm) |
| 1 | Green | Sentinel-2 B3 (560nm) |
| 2 | Red | Sentinel-2 B4 (665nm) |
| 3 | NIR | Sentinel-2 B8 (842nm) |
| 4 | Elevation | Copernicus DEM (meters) |
| 5 | Slope | Terrain slope (degrees) |
| 6 | VegLabel | NDVI-derived vegetation mask (0/1) |

**Next Step:** Proceed to Notebook 2 for model training.